# CC Approval & AECB Data Pull (2025-06 to 2026-06)

Pull credit card numbers approved between Jun 2025 and Jun 2026 from the credit card history view, then retrieve corresponding AECB bureau data for those cases.

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================================
# Step 1: Pull credit cards approved between Jun 2025 and Jun 2026
# ============================================================================
df_cc = spark.sql("""
    SELECT *
    FROM mis_automation.silver_credit_card.vw_credit_card_history
    WHERE APPROVEDATE >= '2025-06-01'
      AND APPROVEDATE <  '2026-07-01'
""")

raw_rows = df_cc.count()
raw_cards = df_cc.select('CREDITCARDNO').distinct().count()
print(f"Before exclusions — Rows: {raw_rows:,} | Unique cards: {raw_cards:,}")

# ============================================================================
# Step 2: Compute per-card OPEN snapshot count (needed for Exclusion 9)
# ============================================================================
w = Window.partitionBy("CREDITCARDNO")
df_cc = df_cc.withColumn(
    "open_count",
    F.sum(F.when(F.col("CURRENT_STATUS") == "1.OPEN", 1).otherwise(0)).over(w)
)

# ============================================================================
# Step 3: Apply business exclusion filters (same as EDA_1)
# ============================================================================
exclude_cond = (
    (F.upper(F.trim(F.col("CLASSCODE"))) == "SME") |                                    # Exclusion 1: SME class code
    (~F.upper(F.trim(F.col("CARD_BUSINESS_SEGMENT"))).isin("CARDS")) |                  # Exclusion 2: Non-CARDS segment
    (F.col("VIP") == 1) |                                                              # Exclusion 3a: VIP customers
    (F.upper(F.trim(F.col("RULING_FAMILY_FLAG"))) == "Y") |                             # Exclusion 3b: Ruling family
    # (F.col("AECB_DATA").isNull()) |                                                  # Exclusion 4: No AECB data (disabled)
    (F.upper(F.trim(F.col("CUSTOMER_SEGMENT"))) != "SALARIED") |                        # Exclusion 6: Non-salaried
    (F.col("PRIMARYCARD") == 0) |                                                      # Exclusion 7: Supplementary cards
    (F.upper(F.trim(F.col("CLASSCODE"))) == "STAFF") |                                  # Exclusion 8: Staff cards
    (F.col("open_count") < 3) |                                                        # Exclusion 9: Fewer than 3 OPEN snapshots
    (F.upper(F.trim(F.col("F_SEGMENT_NEW"))) == "SALARIED-IM") |                        # Exclusion 10a: Salaried-IM
    (F.upper(F.trim(F.col("SEGMENT_POLICY"))).isin(
        "PL CUSTOMERS", "AUTO-LOAN CUSTOMERS", "MORTGAGE CXMS", "STAFF"))               # Exclusion 10b: Unwanted segment policies
)

df_cc_clean = df_cc.filter(~exclude_cond).drop("open_count")

clean_rows = df_cc_clean.count()
clean_cards = df_cc_clean.select('CREDITCARDNO').distinct().count()
clean_refs = df_cc_clean.select('REFERENCE_NUMBER').distinct().count()
print(f"After exclusions  — Rows: {clean_rows:,} | Unique cards: {clean_cards:,} | Unique refs: {clean_refs:,}")
print(f"Excluded: {raw_rows - clean_rows:,} rows, {raw_cards - clean_cards:,} cards")

df_cc_pd = df_cc_clean.toPandas()
print(f"\nApproval date range: {df_cc_pd['APPROVEDATE'].min()} to {df_cc_pd['APPROVEDATE'].max()}")
df_cc_pd.head()

In [0]:
# ============================================================================
# Step 2: Pull AECB bureau data for matching reference numbers
# ============================================================================

# Get unique reference numbers from the credit card pull
ref_numbers = [str(r) for r in df_cc_pd['REFERENCE_NUMBER'].unique().tolist()]
print(f"Unique reference numbers to look up: {len(ref_numbers):,}")

# Load AECB feature store and filter to matching reference numbers
aecb = spark.sql("""
    SELECT *
    FROM mis_automation.silver_bureauone.slv_credit_card_aecb_feature_store
""")

aecb_filtered = (
    aecb
    .withColumn("Reference_Number", F.col("Reference_Number").cast("string"))
    .filter(F.col("Reference_Number").isin(ref_numbers))
)

aecb_pd = aecb_filtered.toPandas()
print(f"AECB rows retrieved: {aecb_pd.shape[0]:,}")
print(f"AECB columns: {aecb_pd.shape[1]}")
print(f"Unique reference numbers in AECB: {aecb_pd['Reference_Number'].nunique():,}")

# Check coverage: how many CC reference numbers have AECB data
matched = set(aecb_pd['Reference_Number'].astype(str)) & set(ref_numbers)
print(f"\nCoverage: {len(matched):,} / {len(ref_numbers):,} reference numbers matched ({len(matched)/len(ref_numbers):.1%})")
aecb_pd.head()

In [0]:
# ============================================================================
# Step 3: Merge AECB data with credit card data & filter to Thin File
# ============================================================================

# Ensure join keys are the same dtype
aecb_pd['Reference_Number'] = aecb_pd['Reference_Number'].astype(str)
df_cc_pd['REFERENCE_NUMBER'] = df_cc_pd['REFERENCE_NUMBER'].astype(str)

# Merge on reference number
df_merged = df_cc_pd.merge(
    aecb_pd,
    left_on='REFERENCE_NUMBER',
    right_on='Reference_Number',
    how='inner'
)
print(f"Merged shape: {df_merged.shape}")
print(f"file_type distribution:\n{df_merged['file_type'].value_counts(dropna=False)}")

# Filter to Thin File only
df_thin = df_merged[df_merged['file_type'] == 'Thin File'].reset_index(drop=True)
print(f"\nThin File shape: {df_thin.shape}")
print(f"Unique cards: {df_thin['CREDITCARDNO'].nunique():,}")
print(f"Unique refs:  {df_thin['REFERENCE_NUMBER'].nunique():,}")
df_thin.head()

In [0]:
# ============================================================================
# Step 4: Define target variable — 90 DPD (delinquency bucket >= 3)
# ============================================================================

# Ensure DELINQUENCYBUCKET is numeric
df_cc_pd['DELINQUENCYBUCKET'] = pd.to_numeric(df_cc_pd['DELINQUENCYBUCKET'], errors='coerce')

# 1. Per card: max delinquency, approval date, reference number
df_target = (
    df_cc_pd
    .groupby('CREDITCARDNO', as_index=False)
    .agg(
        max_delinquency=('DELINQUENCYBUCKET', 'max'),
        APPROVEDATE=('APPROVEDATE', 'first'),
        REFERENCE_NUMBER=('REFERENCE_NUMBER', 'first')
    )
)

# 2. Find the earliest SNAPSHOT_DATE where the max delinquency was first reached
df_worst = (
    df_cc_pd[['CREDITCARDNO', 'DELINQUENCYBUCKET', 'SNAPSHOT_DATE']]
    .merge(df_target[['CREDITCARDNO', 'max_delinquency']], on='CREDITCARDNO')
    .query('DELINQUENCYBUCKET == max_delinquency')
    .sort_values('SNAPSHOT_DATE')
    .drop_duplicates('CREDITCARDNO', keep='first')
    .rename(columns={'SNAPSHOT_DATE': 'worst_snapshot_date'})
    [['CREDITCARDNO', 'worst_snapshot_date']]
)
df_target = df_target.merge(df_worst, on='CREDITCARDNO', how='left')

# 3. Target: 1 (Bad) if max bucket >= 3 (90 DPD), 0 (Good) if == 0, NaN otherwise
df_target['target'] = np.where(
    df_target['max_delinquency'] >= 3, 1,
    np.where(df_target['max_delinquency'] == 0, 0, np.nan)
)

print(f"Total unique cards: {len(df_target):,}")
print(f"\nTarget distribution:")
print(df_target['target'].value_counts(dropna=False))
bad_count = (df_target['target'] == 1).sum()
good_count = (df_target['target'] == 0).sum()
print(f"\nBad rate (90 DPD): {bad_count / (bad_count + good_count):.4%} (excluding indeterminate)")

print(f"\n--- Bad cases (90 DPD) sample ---")
display(df_target[df_target['target'] == 1][['CREDITCARDNO', 'max_delinquency', 'worst_snapshot_date', 'APPROVEDATE', 'REFERENCE_NUMBER', 'target']].head(10))

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# ============================================================================
# Cumulative 90 DPD bad rate by approval-month vintage (MOB curve)
# ============================================================================

# Ensure numeric
df_cc_pd['DELINQUENCYBUCKET'] = pd.to_numeric(df_cc_pd['DELINQUENCYBUCKET'], errors='coerce')

# Derive approval month and months-on-book (MOB)
df_cc_pd['approve_month'] = df_cc_pd['APPROVEDATE'].dt.to_period('M')
df_cc_pd['snapshot_month'] = pd.to_datetime(df_cc_pd['SNAPSHOT_DATE']).dt.to_period('M')
df_cc_pd['mob'] = (df_cc_pd['snapshot_month'] - df_cc_pd['approve_month']).apply(lambda x: x.n if pd.notna(x) else np.nan)

# For each card × snapshot: flag if ever 90 DPD up to that snapshot
df_cc_pd = df_cc_pd.sort_values(['CREDITCARDNO', 'SNAPSHOT_DATE'])
df_cc_pd['cum_max_bucket'] = df_cc_pd.groupby('CREDITCARDNO')['DELINQUENCYBUCKET'].cummax()
df_cc_pd['ever_90dpd'] = (df_cc_pd['cum_max_bucket'] >= 3).astype(int)

# Per vintage × MOB: count distinct cards and distinct bad cards
vintage_mob = (
    df_cc_pd
    .groupby(['approve_month', 'mob'])
    .agg(
        total_cards=('CREDITCARDNO', 'nunique'),
        bad_cards=('ever_90dpd', lambda x: df_cc_pd.loc[x.index].loc[df_cc_pd.loc[x.index, 'ever_90dpd'] == 1, 'CREDITCARDNO'].nunique())
    )
    .reset_index()
)
vintage_mob['bad_rate'] = vintage_mob['bad_cards'] / vintage_mob['total_cards'] * 100

# Plot
fig, ax = plt.subplots(figsize=(14, 7))
for vintage, grp in vintage_mob.sort_values('mob').groupby('approve_month'):
    ax.plot(grp['mob'], grp['bad_rate'], marker='o', markersize=3, label=str(vintage))

ax.set_xlabel('Months on Book (MOB)', fontsize=12)
ax.set_ylabel('Cumulative Bad Rate (90 DPD) %', fontsize=12)
ax.set_title('Cumulative 90 DPD Bad Rate by Approval Month Vintage', fontsize=14)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f%%'))
ax.legend(title='Approval Month', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
# ============================================================================
# Plot 1: Overall bad rate by snapshot month
# Plot 2: Bad rate by snapshot month × approval-month vintage (heatmap)
# ============================================================================

# --- Prep: per card × snapshot, mark if bucket >= 3 at that snapshot ----------
snap_bad = (
    df_cc_pd
    .groupby(['CREDITCARDNO', 'snapshot_month'])
    .agg(max_bucket=('DELINQUENCYBUCKET', 'max'))
    .reset_index()
)
snap_bad['is_bad'] = (snap_bad['max_bucket'] >= 3).astype(int)

# Merge back approve_month
card_vintage = df_cc_pd[['CREDITCARDNO', 'approve_month']].drop_duplicates()
snap_bad = snap_bad.merge(card_vintage, on='CREDITCARDNO')

# --- Plot 1: Overall bad rate by snapshot month -------------------------------
overall = (
    snap_bad
    .groupby('snapshot_month')
    .agg(total=('CREDITCARDNO', 'nunique'),
         bads=('is_bad', 'sum'))
    .reset_index()
)
overall['bad_rate'] = overall['bads'] / overall['total'] * 100
overall = overall.sort_values('snapshot_month')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(overall['snapshot_month'].astype(str), overall['bad_rate'],
        marker='o', color='crimson', linewidth=2)
ax.set_xlabel('Snapshot Month', fontsize=12)
ax.set_ylabel('Bad Rate (90 DPD) %', fontsize=12)
ax.set_title('Overall 90 DPD Bad Rate by Snapshot Month', fontsize=14)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f%%'))
plt.xticks(rotation=45, ha='right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Plot 2: Heatmap – bad rate by vintage × snapshot month -------------------
import seaborn as sns

heat = (
    snap_bad
    .groupby(['approve_month', 'snapshot_month'])
    .agg(total=('CREDITCARDNO', 'nunique'),
         bads=('is_bad', 'sum'))
    .reset_index()
)
heat['bad_rate'] = heat['bads'] / heat['total'] * 100

pivot = heat.pivot(index='approve_month', columns='snapshot_month', values='bad_rate')
pivot.index = pivot.index.astype(str)
pivot.columns = pivot.columns.astype(str)

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Bad Rate %'}, ax=ax)
ax.set_xlabel('Snapshot Month', fontsize=12)
ax.set_ylabel('Approval Month Vintage', fontsize=12)
ax.set_title('90 DPD Bad Rate (%) by Vintage × Snapshot Month', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [0]:
# ============================================================================
# PRIOR YEAR: Pull credit cards approved between Jun 2024 and Jun 2025
# ============================================================================
df_cc_prev = spark.sql("""
    SELECT *
    FROM mis_automation.silver_credit_card.vw_credit_card_history
    WHERE APPROVEDATE >= '2024-06-01'
      AND APPROVEDATE <  '2025-07-01'
""")

raw_rows_prev = df_cc_prev.count()
raw_cards_prev = df_cc_prev.select('CREDITCARDNO').distinct().count()
print(f"Before exclusions — Rows: {raw_rows_prev:,} | Unique cards: {raw_cards_prev:,}")

# Compute per-card OPEN snapshot count
w = Window.partitionBy("CREDITCARDNO")
df_cc_prev = df_cc_prev.withColumn(
    "open_count",
    F.sum(F.when(F.col("CURRENT_STATUS") == "1.OPEN", 1).otherwise(0)).over(w)
)

# Apply same business exclusion filters
exclude_cond_prev = (
    (F.upper(F.trim(F.col("CLASSCODE"))) == "SME") |
    (~F.upper(F.trim(F.col("CARD_BUSINESS_SEGMENT"))).isin("CARDS")) |
    (F.col("VIP") == 1) |
    (F.upper(F.trim(F.col("RULING_FAMILY_FLAG"))) == "Y") |
    (F.upper(F.trim(F.col("CUSTOMER_SEGMENT"))) != "SALARIED") |
    (F.col("PRIMARYCARD") == 0) |
    (F.upper(F.trim(F.col("CLASSCODE"))) == "STAFF") |
    (F.col("open_count") < 3) |
    (F.upper(F.trim(F.col("F_SEGMENT_NEW"))) == "SALARIED-IM") |
    (F.upper(F.trim(F.col("SEGMENT_POLICY"))).isin(
        "PL CUSTOMERS", "AUTO-LOAN CUSTOMERS", "MORTGAGE CXMS", "STAFF"))
)

df_cc_clean_prev = df_cc_prev.filter(~exclude_cond_prev).drop("open_count")

clean_rows_prev = df_cc_clean_prev.count()
clean_cards_prev = df_cc_clean_prev.select('CREDITCARDNO').distinct().count()
clean_refs_prev = df_cc_clean_prev.select('REFERENCE_NUMBER').distinct().count()
print(f"After exclusions  — Rows: {clean_rows_prev:,} | Unique cards: {clean_cards_prev:,} | Unique refs: {clean_refs_prev:,}")
print(f"Excluded: {raw_rows_prev - clean_rows_prev:,} rows, {raw_cards_prev - clean_cards_prev:,} cards")

df_cc_pd_prev = df_cc_clean_prev.toPandas()
print(f"\nApproval date range: {df_cc_pd_prev['APPROVEDATE'].min()} to {df_cc_pd_prev['APPROVEDATE'].max()}")
df_cc_pd_prev.head()

In [0]:
# ============================================================================
# PRIOR YEAR: Pull AECB, merge, filter Thin File
# ============================================================================
ref_numbers_prev = [str(r) for r in df_cc_pd_prev['REFERENCE_NUMBER'].unique().tolist()]
print(f"Unique reference numbers to look up: {len(ref_numbers_prev):,}")

aecb_filtered_prev = (
    aecb
    .withColumn("Reference_Number", F.col("Reference_Number").cast("string"))
    .filter(F.col("Reference_Number").isin(ref_numbers_prev))
)
aecb_pd_prev = aecb_filtered_prev.toPandas()
print(f"AECB rows retrieved: {aecb_pd_prev.shape[0]:,}")
print(f"Unique reference numbers in AECB: {aecb_pd_prev['Reference_Number'].nunique():,}")

# Merge
aecb_pd_prev['Reference_Number'] = aecb_pd_prev['Reference_Number'].astype(str)
df_cc_pd_prev['REFERENCE_NUMBER'] = df_cc_pd_prev['REFERENCE_NUMBER'].astype(str)

df_merged_prev = df_cc_pd_prev.merge(
    aecb_pd_prev, left_on='REFERENCE_NUMBER', right_on='Reference_Number', how='inner'
)
print(f"\nMerged shape: {df_merged_prev.shape}")
print(f"file_type distribution:\n{df_merged_prev['file_type'].value_counts(dropna=False)}")

df_thin_prev = df_merged_prev[df_merged_prev['file_type'] == 'Thin File'].reset_index(drop=True)
print(f"\nThin File shape: {df_thin_prev.shape}")
print(f"Unique cards: {df_thin_prev['CREDITCARDNO'].nunique():,}")

In [0]:
# ============================================================================
# PRIOR YEAR: Define 90 DPD target
# ============================================================================
df_cc_pd_prev['DELINQUENCYBUCKET'] = pd.to_numeric(df_cc_pd_prev['DELINQUENCYBUCKET'], errors='coerce')

df_target_prev = (
    df_cc_pd_prev
    .groupby('CREDITCARDNO', as_index=False)
    .agg(
        max_delinquency=('DELINQUENCYBUCKET', 'max'),
        APPROVEDATE=('APPROVEDATE', 'first'),
        REFERENCE_NUMBER=('REFERENCE_NUMBER', 'first')
    )
)

df_worst_prev = (
    df_cc_pd_prev[['CREDITCARDNO', 'DELINQUENCYBUCKET', 'SNAPSHOT_DATE']]
    .merge(df_target_prev[['CREDITCARDNO', 'max_delinquency']], on='CREDITCARDNO')
    .query('DELINQUENCYBUCKET == max_delinquency')
    .sort_values('SNAPSHOT_DATE')
    .drop_duplicates('CREDITCARDNO', keep='first')
    .rename(columns={'SNAPSHOT_DATE': 'worst_snapshot_date'})
    [['CREDITCARDNO', 'worst_snapshot_date']]
)
df_target_prev = df_target_prev.merge(df_worst_prev, on='CREDITCARDNO', how='left')

df_target_prev['target'] = np.where(
    df_target_prev['max_delinquency'] >= 3, 1,
    np.where(df_target_prev['max_delinquency'] == 0, 0, np.nan)
)

print(f"Total unique cards: {len(df_target_prev):,}")
print(f"\nTarget distribution:")
print(df_target_prev['target'].value_counts(dropna=False))
bad_prev = (df_target_prev['target'] == 1).sum()
good_prev = (df_target_prev['target'] == 0).sum()
print(f"\nBad rate (90 DPD): {bad_prev / (bad_prev + good_prev):.4%} (excluding indeterminate)")
display(df_target_prev[df_target_prev['target'] == 1][['CREDITCARDNO', 'max_delinquency', 'worst_snapshot_date', 'APPROVEDATE', 'REFERENCE_NUMBER', 'target']].head(10))

In [0]:
# ============================================================================
# PRIOR YEAR: Cumulative 90 DPD bad rate by approval-month vintage (MOB)
# ============================================================================
df_cc_pd_prev['approve_month'] = df_cc_pd_prev['APPROVEDATE'].dt.to_period('M')
df_cc_pd_prev['snapshot_month'] = pd.to_datetime(df_cc_pd_prev['SNAPSHOT_DATE']).dt.to_period('M')
df_cc_pd_prev['mob'] = (df_cc_pd_prev['snapshot_month'] - df_cc_pd_prev['approve_month']).apply(lambda x: x.n if pd.notna(x) else np.nan)

df_cc_pd_prev = df_cc_pd_prev.sort_values(['CREDITCARDNO', 'SNAPSHOT_DATE'])
df_cc_pd_prev['cum_max_bucket'] = df_cc_pd_prev.groupby('CREDITCARDNO')['DELINQUENCYBUCKET'].cummax()
df_cc_pd_prev['ever_90dpd'] = (df_cc_pd_prev['cum_max_bucket'] >= 3).astype(int)

vintage_mob_prev = (
    df_cc_pd_prev
    .groupby(['approve_month', 'mob'])
    .agg(
        total_cards=('CREDITCARDNO', 'nunique'),
        bad_cards=('ever_90dpd', lambda x: df_cc_pd_prev.loc[x.index].loc[df_cc_pd_prev.loc[x.index, 'ever_90dpd'] == 1, 'CREDITCARDNO'].nunique())
    )
    .reset_index()
)
vintage_mob_prev['bad_rate'] = vintage_mob_prev['bad_cards'] / vintage_mob_prev['total_cards'] * 100

fig, ax = plt.subplots(figsize=(14, 7))
for vintage, grp in vintage_mob_prev.sort_values('mob').groupby('approve_month'):
    ax.plot(grp['mob'], grp['bad_rate'], marker='o', markersize=3, label=str(vintage))

ax.set_xlabel('Months on Book (MOB)', fontsize=12)
ax.set_ylabel('Cumulative Bad Rate (90 DPD) %', fontsize=12)
ax.set_title('Cumulative 90 DPD Bad Rate by Vintage (2024-06 to 2025-06)', fontsize=14)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f%%'))
ax.legend(title='Approval Month', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
# ============================================================================
# PRIOR YEAR: Bad rate by snapshot month + vintage × snapshot heatmap
# ============================================================================
snap_bad_prev = (
    df_cc_pd_prev
    .groupby(['CREDITCARDNO', 'snapshot_month'])
    .agg(max_bucket=('DELINQUENCYBUCKET', 'max'))
    .reset_index()
)
snap_bad_prev['is_bad'] = (snap_bad_prev['max_bucket'] >= 3).astype(int)

card_vintage_prev = df_cc_pd_prev[['CREDITCARDNO', 'approve_month']].drop_duplicates()
snap_bad_prev = snap_bad_prev.merge(card_vintage_prev, on='CREDITCARDNO')

# --- Plot 1: Overall bad rate by snapshot month -------------------------------
overall_prev = (
    snap_bad_prev
    .groupby('snapshot_month')
    .agg(total=('CREDITCARDNO', 'nunique'), bads=('is_bad', 'sum'))
    .reset_index()
)
overall_prev['bad_rate'] = overall_prev['bads'] / overall_prev['total'] * 100
overall_prev = overall_prev.sort_values('snapshot_month')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(overall_prev['snapshot_month'].astype(str), overall_prev['bad_rate'],
        marker='o', color='crimson', linewidth=2)
ax.set_xlabel('Snapshot Month', fontsize=12)
ax.set_ylabel('Bad Rate (90 DPD) %', fontsize=12)
ax.set_title('Overall 90 DPD Bad Rate by Snapshot Month (2024-06 to 2025-06 Vintages)', fontsize=14)
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f%%'))
plt.xticks(rotation=45, ha='right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Plot 2: Heatmap ----------------------------------------------------------
heat_prev = (
    snap_bad_prev
    .groupby(['approve_month', 'snapshot_month'])
    .agg(total=('CREDITCARDNO', 'nunique'), bads=('is_bad', 'sum'))
    .reset_index()
)
heat_prev['bad_rate'] = heat_prev['bads'] / heat_prev['total'] * 100

pivot_prev = heat_prev.pivot(index='approve_month', columns='snapshot_month', values='bad_rate')
pivot_prev.index = pivot_prev.index.astype(str)
pivot_prev.columns = pivot_prev.columns.astype(str)

fig, ax = plt.subplots(figsize=(18, 9))
sns.heatmap(pivot_prev, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Bad Rate %'}, ax=ax, annot_kws={'size': 8})
ax.set_xlabel('Snapshot Month', fontsize=12)
ax.set_ylabel('Approval Month Vintage', fontsize=12)
ax.set_title('90 DPD Bad Rate (%) by Vintage × Snapshot Month (2024-06 to 2025-06)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()